### Task 2: Text chunking Embedding and Vector store indexing

#### Importing all libraries needed

In [2]:
import pandas as pd
import numpy as np
import os


from langchain_text_splitters import RecursiveCharacterTextSplitter


from sentence_transformers import SentenceTransformer

import chromadb
from chromadb.config import Settings

from tqdm import tqdm

print("All libraries imported successfully!")

All libraries imported successfully!


### Load cleaned Dataset

In [3]:
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
DATA_PATH = os.path.join(BASE_DIR, "data", "processed", "filtered_complaints.csv")

df = pd.read_csv(DATA_PATH)

print(f"Cleaned dataset loaded successfully!")
print(f"Total complaints: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nComplaints per product:")
print(df['product_category'].value_counts())

Cleaned dataset loaded successfully!
Total complaints: 465,679
Columns: ['complaint_id', 'product_category', 'product', 'issue', 'sub_issue', 'company', 'state', 'date_received', 'original_narrative', 'cleaned_narrative']

Complaints per product:
product_category
Credit Card        189334
Savings Account    140319
Money Transfer      98685
Personal Loan       37341
Name: count, dtype: int64


### Stratified Sampling
Creating a representative sample of 12,000 complaints from the full dataset.


In [5]:
SAMPLE_SIZE = 12000

total_complaints = len(df)

print("Stratified sampling plan:")
print(f"Total dataset: {total_complaints:,}")
print(f"Target sample: {SAMPLE_SIZE:,}")
print(f"\nBreakdown per product:")

sample_sizes = {}

for product, count in df['product_category'].value_counts().items():
    proportion = count / total_complaints
    sample_n = round(SAMPLE_SIZE * proportion)
    sample_sizes[product] = sample_n
    print(f"{product:<20} {count:>8,} complaints "
          f"({proportion*100:.1f}%) → sample: {sample_n:,}")

print(f"Total sample:        {sum(sample_sizes.values()):,}")

Stratified sampling plan:
Total dataset: 465,679
Target sample: 12,000

Breakdown per product:
Credit Card           189,334 complaints (40.7%) → sample: 4,879
Savings Account       140,319 complaints (30.1%) → sample: 3,616
Money Transfer         98,685 complaints (21.2%) → sample: 2,543
Personal Loan          37,341 complaints (8.0%) → sample: 962
Total sample:        12,000


In [6]:
sample_dfs = []

for product, n in sample_sizes.items():
    product_df = df[df['product_category'] == product]
    sampled = product_df.sample(n=n, random_state=42)
    sample_dfs.append(sampled)
    print(f"Sampled {n:,} from {product}")

df_sample = pd.concat(sample_dfs, ignore_index=True)

df_sample = df_sample.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nFinal sample shape: {df_sample.shape}")
print(f"\nSample distribution:")
print(df_sample['product_category'].value_counts())

Sampled 4,879 from Credit Card
Sampled 3,616 from Savings Account
Sampled 2,543 from Money Transfer
Sampled 962 from Personal Loan

Final sample shape: (12000, 10)

Sample distribution:
product_category
Credit Card        4879
Savings Account    3616
Money Transfer     2543
Personal Loan       962
Name: count, dtype: int64


### Text Chunking

In [7]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

sample_text = df_sample['cleaned_narrative'].iloc[0]

print("ORIGINAL TEXT:")
print(f"Length: {len(sample_text)} characters")
print(sample_text[:300])

test_chunks = text_splitter.split_text(sample_text)

print(f"\nNumber of chunks produced: {len(test_chunks)}")
print("\nCHUNKS:")
for i, chunk in enumerate(test_chunks):
    print(f"\n--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk)

ORIGINAL TEXT:
Length: 527 characters
i am a loyal customer at amex from 1993 i opened a new amex card and i decided last month to enroll it in auto pay i got the confirmation today i received a note that my account is delinquent and they increased my rate because the auto pay kicks in only after a month in this way for the people busy 

Number of chunks produced: 2

CHUNKS:

--- Chunk 1 (499 chars) ---
i am a loyal customer at amex from 1993 i opened a new amex card and i decided last month to enroll it in auto pay i got the confirmation today i received a note that my account is delinquent and they increased my rate because the auto pay kicks in only after a month in this way for the people busy like myself who do n t read a lengthy autopay confirmation that states this it is a trap to increase their rate this should be stopped and forced these companies to remind you this if you did n t pay

--- Chunk 2 (75 chars) ---
companies to remind you this if you did n t pay the dues before t

### Apply Chunking to all 12,000 Complaints

In [8]:
all_chunks = []

for idx, row in tqdm(df_sample.iterrows(), 
                     total=len(df_sample),
                     desc="Chunking complaints"):
    
    
    narrative = str(row['cleaned_narrative'])
    
    
    if len(narrative) < 50:
        continue
    
    
    chunks = text_splitter.split_text(narrative)
    
    
    for chunk_idx, chunk in enumerate(chunks):
        all_chunks.append({
            'chunk_text': chunk,
            'complaint_id': str(row['complaint_id']),
            'product_category': row['product_category'],
            'issue': str(row['issue']),
            'company': str(row['company']),
            'state': str(row['state']),
            'date_received': str(row['date_received']),
            'chunk_index': chunk_idx,
            'total_chunks': len(chunks)
        })


df_chunks = pd.DataFrame(all_chunks)

print(f"Total complaints processed: {len(df_sample):,}")
print(f"Total chunks produced: {len(df_chunks):,}")
print(f"Average chunks per complaint: {len(df_chunks)/len(df_sample):.1f}")
print(f"\nChunk length statistics:")
df_chunks['chunk_length'] = df_chunks['chunk_text'].apply(len)
print(df_chunks['chunk_length'].describe().round(1))

Chunking complaints: 100%|██████████| 12000/12000 [00:09<00:00, 1272.86it/s]


Total complaints processed: 12,000
Total chunks produced: 32,963
Average chunks per complaint: 2.7

Chunk length statistics:
count    32963.0
mean       412.2
std        134.9
min         45.0
25%        338.0
50%        495.0
75%        498.0
max        500.0
Name: chunk_length, dtype: float64


#### Embedding model

In [9]:
print("Loading the model...")

model = SentenceTransformer('all-MiniLM-L6-v2')

print("Model loaded successfully!")

test_sentence = "My credit card was declined without any reason"
test_embedding = model.encode(test_sentence)

print(f"\nTest sentence: '{test_sentence}'")
print(f"Embedding shape: {test_embedding.shape}")
print(f"First 10 numbers: {test_embedding[:10].round(4)}")
print(f"\nThis is what the AI sees ")
print(f"384 numbers that capture the meaning of the sentence.")

Loading the model...


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded successfully!

Test sentence: 'My credit card was declined without any reason'
Embedding shape: (384,)
First 10 numbers: [ 0.0319  0.0645  0.0414  0.0363 -0.0193 -0.0605  0.0075  0.0181  0.1065
 -0.0806]

This is what the AI sees 
384 numbers that capture the meaning of the sentence.


#### Generate embeddings for all chunks

In [10]:
chunk_texts = df_chunks['chunk_text'].tolist()

print(f"Generating embeddings for {len(chunk_texts):,} chunks...")
print("This will take 3-8 minutes on a standard laptop...\n")


embeddings = model.encode(
    chunk_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\nEmbeddings generated successfully!")
print(f"Embeddings shape: {embeddings.shape}")
print(f"This means: {embeddings.shape[0]:,} chunks × {embeddings.shape[1]} dimensions")

Generating embeddings for 32,963 chunks...
This will take 3-8 minutes on a standard laptop...



Batches:   0%|          | 0/516 [00:00<?, ?it/s]


Embeddings generated successfully!
Embeddings shape: (32963, 384)
This means: 32,963 chunks × 384 dimensions


### Vector Store with ChromaDB

In [11]:
VECTOR_STORE_PATH = os.path.join(BASE_DIR, "vector_store")

client = chromadb.PersistentClient(path=VECTOR_STORE_PATH)

try:
    client.delete_collection("complaints")
    print("Deleted existing collection — starting fresh")
except:
    print("No existing collection found — creating new one")

collection = client.create_collection(
    name="complaints",
    metadata={"hnsw:space": "cosine"}
)

print(f"ChromaDB collection created!")
print(f"Saved at: {VECTOR_STORE_PATH}")
print(f"Collection name: complaints")
print(f"Distance metric: cosine similarity")

No existing collection found — creating new one
ChromaDB collection created!
Saved at: c:\Users\HP\Desktop\Week-7\vector_store
Collection name: complaints
Distance metric: cosine similarity


### Add chunks to ChromaDB in batches

In [12]:
BATCH_SIZE = 500
total_chunks = len(df_chunks)
num_batches = (total_chunks // BATCH_SIZE) + 1

print(f"Adding {total_chunks:,} chunks to ChromaDB...")
print(f"Batch size: {BATCH_SIZE}")
print(f"Number of batches: {num_batches}\n")

for batch_num in tqdm(range(num_batches), desc="Indexing into ChromaDB"):
    
    
    start = batch_num * BATCH_SIZE
    end = min(start + BATCH_SIZE, total_chunks)
    
    
    if start >= total_chunks:
        break
    
    
    batch_df = df_chunks.iloc[start:end]
    batch_embeddings = embeddings[start:end]
    
   
    collection.add(
        ids=[f"chunk_{i}" for i in range(start, end)],
        embeddings=batch_embeddings.tolist(),
        documents=batch_df['chunk_text'].tolist(),
        metadatas=batch_df[[
            'complaint_id',
            'product_category', 
            'issue',
            'company',
            'state',
            'date_received',
            'chunk_index',
            'total_chunks'
        ]].to_dict('records')
    )

print(f"\nAll chunks indexed successfully!")
print(f"Total chunks in ChromaDB: {collection.count():,}")

Adding 32,963 chunks to ChromaDB...
Batch size: 500
Number of batches: 66



Indexing into ChromaDB: 100%|██████████| 66/66 [03:11<00:00,  2.90s/it]



All chunks indexed successfully!
Total chunks in ChromaDB: 32,963


### Verify the vector store

In [18]:
test_query = "Why are customers unhappy with credit card charges?"

print(f"Test query: '{test_query}'")



query_embedding = model.encode(test_query).tolist()


results = collection.query(
    query_embeddings=[query_embedding],
    n_results=2,
    include=['documents', 'metadatas', 'distances']
)


print(f"\nTop 2 most relevant chunks:\n")

for i in range(2):
    print(f"--- Result {i+1} ---")
    print(f"Product:     {results['metadatas'][0][i]['product_category']}")
    print(f"Issue:       {results['metadatas'][0][i]['issue']}")
    print(f"Company:     {results['metadatas'][0][i]['company']}")
    print(f"Similarity:  {1 - results['distances'][0][i]:.4f}")
    print(f"Text:        {results['documents'][0][i][:200]}...")
    print()

Test query: 'Why are customers unhappy with credit card charges?'

Top 2 most relevant chunks:

--- Result 1 ---
Product:     Credit Card
Issue:       Advertising and marketing, including promotional offers
Company:     SYNCHRONY FINANCIAL
Similarity:  0.6537
Text:        let alone the statements do not have transparency around what the promotional periods are and what the fees are if they are not paid off this is so predatory the sales staff was not equipped to sell t...

--- Result 2 ---
Product:     Credit Card
Issue:       Problem with a purchase shown on your statement
Company:     CITIBANK, N.A.
Similarity:  0.6466
Text:        i did not do if they made the card then someone other one can make it to i have been a customer with this company since years x x dollars of business and this is how they treat me and have never misse...



### Test with product filter

In [19]:
print("Filtered search — Credit Card complaints only")


results_filtered = collection.query(
    query_embeddings=[query_embedding],
    n_results=2,
    where={"product_category": "Credit Card"},
    include=['documents', 'metadatas', 'distances']
)

print(f"\nTop 2 Credit Card chunks for: '{test_query}'\n")

for i in range(2):
    print(f"--- Result {i+1} ---")
    print(f"Product:    {results_filtered['metadatas'][0][i]['product_category']}")
    print(f"Issue:      {results_filtered['metadatas'][0][i]['issue']}")
    print(f"Similarity: {1 - results_filtered['distances'][0][i]:.4f}")
    print(f"Text:       {results_filtered['documents'][0][i][:200]}...")
    print()

Filtered search — Credit Card complaints only

Top 2 Credit Card chunks for: 'Why are customers unhappy with credit card charges?'

--- Result 1 ---
Product:    Credit Card
Issue:      Advertising and marketing, including promotional offers
Similarity: 0.6537
Text:       let alone the statements do not have transparency around what the promotional periods are and what the fees are if they are not paid off this is so predatory the sales staff was not equipped to sell t...

--- Result 2 ---
Product:    Credit Card
Issue:      Problem with a purchase shown on your statement
Similarity: 0.6466
Text:       i did not do if they made the card then someone other one can make it to i have been a customer with this company since years x x dollars of business and this is how they treat me and have never misse...



## Summary

### Sampling Strategy
 used proportional stratified random sampling with a fixed random 
seed of 42 for reproducibility. 12,000 complaints were selected 
maintaining exact proportions across all 4 product categories.

### Chunking Approach
RecursiveCharacterTextSplitter with chunk_size=500 and chunk_overlap=50.
12,000 complaints produced 32,963 chunks averaging 2.7 chunks per complaint.
The recursive splitter preserves sentence boundaries better than 
simple character splitting.

### Embedding Model Choice
all-MiniLM-L6-v2 was chosen for its balance of speed, size (~80MB), 
and quality. It produces 384-dimensional embeddings optimized for 
semantic similarity — exactly what RAG retrieval requires.
The model is free, open source, and industry standard for RAG pipelines.